# Notebook_C (dùng chung, phần dữ liệu, Sinh viên A): thu thập, so sánh nguồn, làm sạch, EDA và SQL

**Dùng chung cho mọi đề tài của môn.** Chỉ cần điền khối CONFIG bên dưới (tên đề tài, nguồn, tên cột, tần suất) rồi chạy tuần tự. Mọi ô code có chú thích tiếng Anh; phần giải thích, lý do và cách đọc kết quả bằng tiếng Việt nằm ngay trên mỗi ô.

**Sản phẩm bàn giao cho Sinh viên B**: `data/processed/feat.parquet`, `data/processed/manifest.json`, thư mục `report/` (bảng RQ1, hình, nhật ký làm sạch, AI Audit Log).

**Ba câu hỏi nghiên cứu chung (áp dụng cho mọi đề, mỗi nhóm thêm 1-2 câu riêng theo đề):**

* **RQ1 (dữ liệu và mô tả, sâu hơn)**: Hai nguồn dữ liệu (chính và phụ) có nhất quán với nhau không, biến mục tiêu biến động theo mùa, theo đơn vị và theo biến phụ như thế nào, và phần nào của biến động được giải thích bằng lịch so với bằng nguồn phụ?
* **RQ2 (mô hình và tổng quát hoá)**: Trong sáu họ mô hình (mốc naive, ridge, random forest, LightGBM toàn cục, tuyến tính dài hạn, mô hình nền zero-shot), mô hình nào tốt nhất theo tầm dự báo, và kết quả có giữ được khi lùi gốc dự báo theo thời gian và khi gặp đơn vị chưa thấy không?
* **RQ3 (tin cậy và vận hành)**: Khoảng tin cậy có đúng bao phủ không, cảnh báo sự kiện (mục tiêu vượt phân vị cao) đạt recall và precision bao nhiêu, sai số thay đổi thế nào khi chế độ đổi, và nguồn phụ đóng góp bao nhiêu (ablation)?

Notebook_C trả lời RQ1 và chuẩn bị dữ liệu cho RQ2, RQ3 (Notebook_D).

In [6]:
TOPIC = "Internet Traffic Forecasting and Outage Warning across Twenty Countries from Cloudflare Radar with Cloud Incident Covariates"
GROUP = "3"
PRIMARY_SOURCE = {"name": "Cloudflare Radar API (netflows/timeseries + annotations/outages)",
                   "url": "https://radar.cloudflare.com/",
                   "license": "Free via personal API Token; no public redistribution of raw data (Radar API Terms of Service)",
                   "path": "data/raw/primary.csv"}
SECOND_SOURCE  = {"name": "cloud_incidents.csv (self-compiled from official AWS/Azure/GCP status pages)",
                   "url": "https://health.aws.amazon.com/health/status , https://azure.status.microsoft/ , https://status.cloud.google.com/",
                   "license": "Manually compiled from public reports, with sources cited",
                   "path": "data/raw/secondary.csv"}
UNIT_COL   = "loc"
TIME_COL   = "ts"
TARGET_COL = "traffic"
EXOG_COLS  = ["cloud_inc"]
FREQ       = "h"
HORIZONS   = [1, 24]
SEASON     = 168
TEST_START = "2026-04-01"
EVENT_QUANTILE = 0.9
print("Topic:", TOPIC); print("Group:", GROUP)

Topic: Internet Traffic Forecasting and Outage Warning across Twenty Countries from Cloudflare Radar with Cloud Incident Covariates
Group: 3


In [3]:
import os
os.chdir(r'D:\Semester3\ADY201m\Project')
import pandas as pd

radar = pd.read_parquet('data/processed/radar.parquet')
grid = radar[['ts']].drop_duplicates().sort_values('ts').reset_index(drop=True)

inc = pd.read_csv('data/raw/cloud_incidents.csv', parse_dates=['start', 'end'])

grid['cloud_inc'] = 0
for r in inc.itertuples():
    grid.loc[(grid.ts >= r.start) & (grid.ts <= r.end), 'cloud_inc'] = 1

grid.to_csv('data/raw/secondary.csv', index=False)
print('secondary.csv created:', grid.shape)
print('cloud_inc share:', grid.cloud_inc.mean())

secondary.csv created: (26303, 2)
cloud_inc share: 0.010150933353609854


In [2]:
import os
os.chdir(r'D:\Semester3\ADY201m\Project')
import pandas as pd
radar = pd.read_parquet('data/processed/radar.parquet')
primary = radar[['loc', 'ts', 'traffic', 'outage']].copy()
primary.to_csv('data/raw/primary.csv', index=False)
print('primary.csv created:', primary.shape)

primary.csv created: (526060, 4)


In [8]:
# AI Audit Log helper: every prompt that changed your work is one row (2 minutes per entry, 3-5 per week)
import pandas as pd, os, datetime as dt
os.makedirs("report", exist_ok=True)
AUDIT_PATH = "report/ai_audit_log.csv"
def audit(step, prompt, tool, ai_output_summary, verified_how, decision, hallucination=False):
    """Append one entry. decision: what you kept / changed / rejected. hallucination=True if the AI answer was wrong and you caught it."""
    row = {"date": dt.date.today().isoformat(), "group": GROUP, "step": step, "prompt": prompt[:500], "tool": tool,
           "ai_output": ai_output_summary[:500], "verified_how": verified_how[:300], "decision": decision[:300], "hallucination": int(hallucination)}
    df = pd.DataFrame([row])
    df.to_csv(AUDIT_PATH, mode="a", header=not os.path.exists(AUDIT_PATH), index=False)
    print("audit entry saved:", step)
# example (delete after reading): audit("Step 2", "Given columns ... how to treat gaps longer than 3 steps?", "Claude", "suggested interpolate(limit=3) then drop", "checked share of gaps > 3 in Q4 below", "kept limit=3, dropped 1.2% rows")

## Hướng dẫn hỏi AI (điền đề tài của nhóm vào chỗ trống)

Quy tắc: hỏi **cụ thể** (kèm tên cột, kích thước, thông báo lỗi), yêu cầu AI **giải thích lý do** và **nêu cách kiểm tra**, rồi tự kiểm tra trước khi dùng. Mỗi prompt làm thay đổi bài phải ghi vào AI Audit Log bằng hàm `audit(...)` ở ô trên. Mẫu prompt theo bước (thay `{TOPIC}`, `{cột}` bằng thông tin thật của nhóm):

| Bước | Mẫu prompt | Cần tự kiểm tra gì |
|---|---|---|
| Thu thập | "Đề tài của tôi là {TOPIC}. Nguồn chính là {PRIMARY_SOURCE}. Hãy gợi ý 3 nguồn phụ công khai (thời tiết, sự kiện, giá, lịch) có thể ghép theo cột {TIME_COL} và {UNIT_COL}, kèm URL tải và giấy phép." | Mở URL, xác nhận có tải được và giấy phép cho phép dùng |
| So sánh nguồn | "Tôi có hai nguồn về cùng biến {TARGET_COL} ở tần suất {FREQ}. Hãy đề xuất 4 chỉ số để so sánh độ phủ, độ trễ cập nhật và độ lệch giữa hai nguồn." | Chạy lại số trên dữ liệu thật, không dùng số AI đưa |
| Làm sạch | "Cột {cột} có {x}% thiếu, thiếu theo cụm dài nhất {n} bước. Nên nội suy hay bỏ? Giải thích rủi ro rò rỉ tương lai." | Kiểm tra nội suy không dùng giá trị tương lai |
| SQL | "Viết truy vấn DuckDB tạo đặc trưng trễ {SEASON} bước và trung bình trượt theo {UNIT_COL}, có WINDOW, không rò rỉ." | Đếm dòng, kiểm tra cột NULL ở đầu chuỗi |
| EDA | "Từ bảng thống kê sau (dán bảng), nêu 3 nhận xét có thể kiểm chứng và 2 điều cần cảnh giác." | Mỗi nhận xét phải chỉ ra được ô số liệu tương ứng |
| Lỗi | "Lỗi: {dán nguyên văn}. Ngữ cảnh: {ô code}. Nguyên nhân và cách sửa tối thiểu?" | Sửa xong chạy lại ô test |

Ví dụ ghi Audit Log sau khi hỏi: `audit("Thu thập", "Đề tài ... gợi ý nguồn phụ", "Claude", "3 nguồn: NOAA ISD, ...", "mở 3 URL, 1 URL lỗi 404", "dùng NOAA ISD, loại nguồn 404", hallucination=True)`.

## Bước 0: môi trường và cấu trúc thư mục

**Làm gì**: tạo môi trường ảo một lần, cài thư viện, tạo thư mục chuẩn. **Vì sao**: cả hai sinh viên và giảng viên chạy cùng phiên bản thư viện thì kết quả tái tạo được; `requirements.txt` là bằng chứng.

**Cấu trúc**: `data/raw` (file gốc, không sửa), `data/processed` (parquet sạch), `sql/` (truy vấn), `report/` (bảng, hình, nhật ký), `notebooks/`.

In [7]:
# Step 0a: run once in a terminal (not in the notebook)
# python -m venv .venv && source .venv/bin/activate      (Windows: .venv\Scripts\activate)
# pip install pandas numpy duckdb pyarrow scikit-learn lightgbm statsmodels matplotlib seaborn ydata-profiling mapie shap requests
# pip freeze > requirements.txt
import os
for d in ["data/raw", "data/processed", "sql", "report", "notebooks"]: os.makedirs(d, exist_ok=True)
print("folders ready")

folders ready


In [9]:
# Step 0b: imports and plotting style (English labels in figures, no top/right spines, no in-figure titles)
import pandas as pd, numpy as np, duckdb, json, hashlib, warnings, re
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore"); np.random.seed(42)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
TEAL, ACC, GREY = "#1B6B6D", "#F4A261", "#9FBFBF"
def style(ax): ax.spines[["top", "right"]].set_visible(False)
def savefig(fig, name): fig.tight_layout(); fig.savefig(f"report/{name}.png", dpi=300); plt.close(fig); print("saved report/" + name + ".png")
con = duckdb.connect()
print("pandas", pd.__version__, "duckdb", duckdb.__version__)

pandas 2.3.3 duckdb 1.5.5


## Bước 1: bài toán, nguồn dữ liệu và kế hoạch thu thập

**Làm gì**: viết vấn đề bằng 5 câu, đăng ký hai nguồn (chính và phụ) vào sổ nguồn (`report/data_sources.csv`) với URL, giấy phép, ngày tải, số dòng. **Vì sao**: phần Data của bài báo cần ghi rõ nguồn là công trình nào, năm nào, số dòng, số cột và lý do chọn; sổ nguồn giúp viết phần đó trong hai phút.

**Nguồn phụ nên chọn**: biến có cơ chế ảnh hưởng rõ đến mục tiêu (thời tiết với nhu cầu, giá với sạc, sự kiện với lưu lượng), có cùng hoặc cao hơn tần suất mục tiêu, và có thể ghép theo thời gian và đơn vị.

In [10]:
# Step 1a: source registry (append every file you download; this becomes Table "Data sources" in the paper)
SOURCES_PATH = "report/data_sources.csv"
def register_source(src, rows=None, cols=None, start=None, end=None, note=""):
    """src: dict with name, url, license, path. rows/cols/start/end are filled after loading."""
    row = {**src, "downloaded": pd.Timestamp.today().date().isoformat(), "rows": rows, "cols": cols, "start": start, "end": end, "note": note}
    df = pd.DataFrame([row]); df.to_csv(SOURCES_PATH, mode="a", header=not os.path.exists(SOURCES_PATH), index=False)
    return row
print("registry at", SOURCES_PATH)

registry at report/data_sources.csv


In [11]:
# Step 1b: load the primary source (edit the reader to match your file: csv, parquet, or an API export saved to data/raw)
def load_table(path):
    """Read csv/parquet with DuckDB so large files never need to fit in memory; returns a pandas DataFrame."""
    if path.endswith(".parquet"): return con.execute(f"SELECT * FROM read_parquet('{path}')").df()
    return con.execute(f"SELECT * FROM read_csv_auto('{path}', union_by_name=true, sample_size=-1)").df()
raw = load_table(PRIMARY_SOURCE["path"])
raw.columns = [c.strip() for c in raw.columns]
raw[TIME_COL] = pd.to_datetime(raw[TIME_COL], errors="coerce", utc=False)
print(raw.shape); display(raw.head(3)); display(raw.dtypes.value_counts())
register_source(PRIMARY_SOURCE, rows=len(raw), cols=raw.shape[1], start=str(raw[TIME_COL].min()), end=str(raw[TIME_COL].max()))

(526060, 4)


,loc,ts,traffic,outage
0,AE,2023-09-15 08:00:00,0.280963,0
1,AE,2023-09-15 09:00:00,0.342169,0
2,AE,2023-09-15 10:00:00,0.433588,0


object            1
datetime64[us]    1
float64           1
int64             1
Name: count, dtype: int64

{'name': 'Cloudflare Radar API (netflows/timeseries + annotations/outages)',
 'url': 'https://radar.cloudflare.com/',
 'license': 'Free via personal API Token; no public redistribution of raw data (Radar API Terms of Service)',
 'path': 'data/raw/primary.csv',
 'downloaded': '2026-09-16',
 'rows': 526060,
 'cols': 4,
 'start': '2023-09-15 08:00:00',
 'end': '2026-09-15 06:00:00',
 'note': ''}

In [12]:
# Step 1c: load the secondary source (covariates); it must share TIME_COL (and UNIT_COL if it is unit-specific)
sec = load_table(SECOND_SOURCE["path"]); sec.columns = [c.strip() for c in sec.columns]
sec[TIME_COL] = pd.to_datetime(sec[TIME_COL], errors="coerce")
SEC_HAS_UNIT = UNIT_COL in sec.columns
print(sec.shape, "unit-specific:", SEC_HAS_UNIT); display(sec.head(3))
register_source(SECOND_SOURCE, rows=len(sec), cols=sec.shape[1], start=str(sec[TIME_COL].min()), end=str(sec[TIME_COL].max()))

(26303, 2) unit-specific: False


,ts,cloud_inc
0,2023-09-15 08:00:00,0
1,2023-09-15 09:00:00,0
2,2023-09-15 10:00:00,0


{'name': 'cloud_incidents.csv (self-compiled from official AWS/Azure/GCP status pages)',
 'url': 'https://health.aws.amazon.com/health/status , https://azure.status.microsoft/ , https://status.cloud.google.com/',
 'license': 'Manually compiled from public reports, with sources cited',
 'path': 'data/raw/secondary.csv',
 'downloaded': '2026-09-16',
 'rows': 26303,
 'cols': 2,
 'start': '2023-09-15 08:00:00',
 'end': '2026-09-15 06:00:00',
 'note': ''}

### Xuất và đọc lại dữ liệu bằng CSV

**Quy ước file**: mọi bước lưu một file CSV riêng để giảng viên và bạn cùng nhóm mở được bằng Excel mà không cần Python: `data/raw/primary_snapshot.csv` (bản gốc đã chuẩn tên cột và kiểu thời gian, không sửa giá trị), `data/processed/clean.csv` (sau làm sạch, một dòng một đơn vị-thời điểm), `data/processed/feat.csv` (bảng đặc trưng bàn giao). Parquet vẫn lưu song song vì nhanh và giữ đúng kiểu dữ liệu; CSV là bản để đọc và nộp. Khi đọc lại CSV, luôn ép kiểu thời gian bằng `parse_dates` và kiểm tra số dòng trùng khớp với manifest.

In [ ]:
# Export the standardised raw snapshot to CSV and read it back to confirm the round trip (dtypes and row count)
RAW_CSV = "data/raw/primary_snapshot.csv"
raw.to_csv(RAW_CSV, index=False)
check = pd.read_csv(RAW_CSV, parse_dates=[TIME_COL])
assert len(check) == len(raw), "row count changed in the CSV round trip"
print("wrote", RAW_CSV, "| rows", len(check), "| columns", list(check.columns)[:8])

### Bước 1d: thu thập thêm dữ liệu (mở rộng phạm vi thời gian hoặc đơn vị)

**Làm gì**: nếu nguồn cho phép, tải thêm giai đoạn hoặc thêm đơn vị (trạm, vùng) để bảng đủ khoảng 100 nghìn dòng và có đủ ít nhất hai mùa hoặc hai chế độ. Hàm dưới ghép nhiều file tải theo năm hoặc theo đơn vị và loại trùng. **Vì sao**: mô hình toàn cục và kiểm thử đơn vị chưa thấy cần nhiều đơn vị; đổi chế độ chỉ đo được khi có dữ liệu trước và sau.

In [ ]:
# Step 1d: combine several downloaded chunks (data/raw/primary_*.csv) into one table, deduplicated on unit and time
import glob
chunks = sorted(glob.glob(PRIMARY_SOURCE["path"].replace(".csv", "_*.csv")))
if chunks:
    extra = pd.concat([load_table(f) for f in chunks]); extra[TIME_COL] = pd.to_datetime(extra[TIME_COL], errors="coerce")
    before = len(raw); raw = pd.concat([raw, extra]).drop_duplicates([UNIT_COL, TIME_COL] if UNIT_COL in raw.columns else [TIME_COL])
    print(f"added {len(raw) - before} rows from {len(chunks)} chunks")
else:
    print("no extra chunks found (pattern primary_*.csv); skip if your source is a single file")
if UNIT_COL not in raw.columns: raw[UNIT_COL] = "all"; print("single-series data: UNIT_COL set to 'all'")

## Bước 2: so sánh hai nguồn và làm sạch có nhật ký

**So sánh nguồn (RQ1a)**: trước khi ghép, đo (1) khoảng thời gian chồng nhau, (2) tỷ lệ bước thời gian có ở nguồn này mà không có ở nguồn kia, (3) nếu hai nguồn có cùng một biến thì tương quan và độ lệch trung bình, (4) độ trễ cập nhật (thời điểm cuối của mỗi nguồn). Bảng này vào phần Data của bài.

**Làm sạch có nhật ký**: mỗi thao tác ghi số dòng trước, sau và lý do; bảng nhật ký là bằng chứng để người đọc tin dữ liệu.

In [ ]:
# Step 2a: source comparison table
def compare_sources(a, b, time_col, common_var=None):
    ta, tb = a[time_col].dropna(), b[time_col].dropna()
    ov_start, ov_end = max(ta.min(), tb.min()), min(ta.max(), tb.max())
    ga = set(ta.dt.floor(FREQ).unique()); gb = set(tb.dt.floor(FREQ).unique())
    rows = [["overlap start", ov_start], ["overlap end", ov_end], ["steps only in primary", len(ga - gb)], ["steps only in secondary", len(gb - ga)],
            ["latest timestamp primary", ta.max()], ["latest timestamp secondary", tb.max()]]
    if common_var and common_var in a.columns and common_var in b.columns:
        m = a.groupby(time_col)[common_var].mean().to_frame("a").join(b.groupby(time_col)[common_var].mean().to_frame("b"), how="inner")
        rows += [["correlation of common variable", round(m.a.corr(m.b), 3)], ["mean difference (primary minus secondary)", round((m.a - m.b).mean(), 3)]]
    return pd.DataFrame(rows, columns=["metric", "value"])
cmp_tbl = compare_sources(raw, sec, TIME_COL, common_var=None)     # set common_var="temp" if both sources carry the same variable
cmp_tbl.to_csv("report/table_source_comparison.csv", index=False); display(cmp_tbl)

In [ ]:
# Step 2b: cleaning with a log; each step records rows before and after and the reason
clean_log = []
def step(df, name, fn, reason):
    n0 = len(df); out = fn(df); clean_log.append([name, n0, len(out), n0 - len(out), reason]); return out
df = raw.copy()
df = step(df, "parse time", lambda x: x.dropna(subset=[TIME_COL]), "rows with unparseable timestamps")
df = step(df, "drop duplicates", lambda x: x.drop_duplicates([UNIT_COL, TIME_COL]), "same unit and timestamp")
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = step(df, "drop missing target", lambda x: x.dropna(subset=[TARGET_COL]), "target missing")
lo, hi = df[TARGET_COL].quantile([0.001, 0.999])
df = step(df, "physical range", lambda x: x[(x[TARGET_COL] >= lo) & (x[TARGET_COL] <= hi)], f"outside [{lo:.3g}, {hi:.3g}] (0.1% tails; replace with a physical range if you know it)")
pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]).to_csv("report/table_cleaning_log.csv", index=False)
display(pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]))

In [ ]:
# Step 2c: regular time grid per unit, then short-gap interpolation (limit = 3 steps, never across long gaps)
df = df.set_index(TIME_COL).groupby(UNIT_COL)[[TARGET_COL] + [c for c in df.columns if c not in (UNIT_COL, TIME_COL, TARGET_COL) and pd.api.types.is_numeric_dtype(df[c])]].resample(FREQ).mean().reset_index()
gap = df.groupby(UNIT_COL)[TARGET_COL].apply(lambda s: s.isna().mean()).rename("missing_share")
df[TARGET_COL] = df.groupby(UNIT_COL)[TARGET_COL].transform(lambda s: s.interpolate(limit=3))
n_before = len(df); df = df.dropna(subset=[TARGET_COL]); clean_log.append(["regular grid + interpolate(limit=3)", n_before, len(df), n_before - len(df), "gaps longer than 3 steps dropped"])
print("missing share per unit before interpolation:"); display(gap.describe().round(3))

In [ ]:
# Step 2d: keep units with enough history (at least 90% of the longest unit) so the global model is not dominated by fragments
cnt = df.groupby(UNIT_COL).size(); keep = cnt[cnt >= 0.9 * cnt.max()].index
n_before = len(df); df = df[df[UNIT_COL].isin(keep)]; clean_log.append(["drop short units", n_before, len(df), n_before - len(df), f"{len(cnt) - len(keep)} units with < 90% coverage"])
print("units kept:", len(keep), "of", len(cnt))

In [ ]:
# Step 2e: merge the secondary source (nearest timestamp within one step; by unit if the secondary source is unit-specific)
sec_num = sec[[TIME_COL] + ([UNIT_COL] if SEC_HAS_UNIT else []) + [c for c in EXOG_COLS if c in sec.columns]].copy()
for c in EXOG_COLS:
    if c in sec_num.columns: sec_num[c] = pd.to_numeric(sec_num[c], errors="coerce")
sec_num = sec_num.sort_values(TIME_COL); df = df.sort_values(TIME_COL)
tol = pd.Timedelta(pd.tseries.frequencies.to_offset(FREQ))
m = pd.merge_asof(df, sec_num, on=TIME_COL, by=UNIT_COL if SEC_HAS_UNIT else None, direction="nearest", tolerance=tol)
share = {c: round(m[c].notna().mean(), 3) for c in EXOG_COLS if c in m.columns}
print("share of rows with each covariate:", share); df = m.sort_values([UNIT_COL, TIME_COL]).reset_index(drop=True)

In [ ]:
# Step 2f: save the clean table and the cleaning log; print a one-paragraph data statement for the paper
df.to_parquet("data/processed/clean.parquet", index=False)
df.to_csv("data/processed/clean.csv", index=False)                     # separate clean CSV for submission and for opening in Excel
chk = pd.read_csv("data/processed/clean.csv", parse_dates=[TIME_COL]); assert len(chk) == len(df) and chk[TARGET_COL].notna().all(), "clean.csv round trip failed"
print("wrote data/processed/clean.csv with", len(chk), "rows")
pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]).to_csv("report/table_cleaning_log.csv", index=False)
print(f"Data statement: {PRIMARY_SOURCE['name']} ({PRIMARY_SOURCE['license']}) merged with {SECOND_SOURCE['name']}; {len(df):,} rows, {df[UNIT_COL].nunique()} units, "
      f"{df[TIME_COL].min().date()} to {df[TIME_COL].max().date()} at frequency {FREQ}; {sum(r[3] for r in clean_log):,} rows removed by cleaning.")

## EDA (RQ1): mô tả, mùa vụ, đơn vị, nguồn phụ, tự tương quan

**Cách đọc**: mỗi hình trả lời một câu hỏi nhỏ và ghi một nhận xét có thể kiểm chứng trong caption. Không vẽ hình chỉ để đẹp. Thứ tự: (1) tổng quan phân bố, (2) chuỗi theo thời gian của vài đơn vị, (3) mùa vụ theo lịch, (4) khác biệt giữa đơn vị, (5) quan hệ với nguồn phụ và độ trễ, (6) tự tương quan quyết định đặc trưng trễ, (7) thiếu dữ liệu.

In [ ]:
# EDA 1: automatic profile (open report/profile.html in a browser) and summary statistics
from ydata_profiling import ProfileReport
sample = df.sample(min(20000, len(df)), random_state=42)
ProfileReport(sample, title=f"Profile {GROUP}", minimal=True).to_file("report/profile.html")
desc = df[[TARGET_COL] + [c for c in EXOG_COLS if c in df.columns]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.round(3)
desc.to_csv("report/table_describe.csv"); display(desc)

In [ ]:
# EDA 2: distribution of the target (histogram with many bins) and log-scale check for skewed data
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].hist(df[TARGET_COL], bins=60, color=TEAL); axes[0].set_xlabel(TARGET_COL); axes[0].set_ylabel("Count"); style(axes[0])
pos = df[TARGET_COL][df[TARGET_COL] > 0]
axes[1].hist(np.log1p(pos), bins=60, color=ACC); axes[1].set_xlabel("log1p(" + TARGET_COL + ")"); style(axes[1])
savefig(fig, "fig_eda_distribution"); print("skewness:", round(df[TARGET_COL].skew(), 2), "-> consider modelling log1p if skewness > 2")

In [ ]:
# EDA 3: time series of the first four units over the whole period (look for trends, regime changes, gaps)
units = df[UNIT_COL].unique()[:4]
fig, ax = plt.subplots(figsize=(11, 3.4))
for u, col in zip(units, [TEAL, ACC, GREY, "#264653"]):
    g = df[df[UNIT_COL] == u]; ax.plot(g[TIME_COL], g[TARGET_COL], lw=0.7, color=col, label=str(u))
ax.axvline(pd.Timestamp(TEST_START), color="k", ls="--", lw=1); ax.set_ylabel(TARGET_COL); ax.legend(frameon=False, ncol=4); style(ax)
savefig(fig, "fig_eda_timeseries")

In [ ]:
# EDA 4: calendar seasonality: mean target by hour of day, day of week and month (only the ones that make sense at your FREQ)
t = df[TIME_COL]; cal = pd.DataFrame({"hour": t.dt.hour, "dow": t.dt.dayofweek, "month": t.dt.month, TARGET_COL: df[TARGET_COL]})
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, k in zip(axes, ["hour", "dow", "month"]):
    s = cal.groupby(k)[TARGET_COL].mean(); ax.plot(s.index, s.values, marker="o", color=TEAL); ax.set_xlabel(k); style(ax)
axes[0].set_ylabel("mean " + TARGET_COL); savefig(fig, "fig_eda_seasonality")
season_strength = {k: round(cal.groupby(k)[TARGET_COL].mean().std() / cal[TARGET_COL].std(), 3) for k in ["hour", "dow", "month"]}
print("seasonal strength (std of group means / total std):", season_strength)

In [ ]:
# EDA 5: heterogeneity across units: mean and variability per unit, sorted (global models must handle this spread)
per_unit = df.groupby(UNIT_COL)[TARGET_COL].agg(["mean", "std", "min", "max", "count"]).sort_values("mean")
per_unit.to_csv("report/table_per_unit.csv")
fig, ax = plt.subplots(figsize=(10, 3.2)); ax.bar(range(len(per_unit)), per_unit["mean"], color=TEAL, yerr=None)
ax.set_xlabel("units sorted by mean"); ax.set_ylabel("mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_units")
print("ratio of largest to smallest unit mean:", round(per_unit["mean"].max() / max(per_unit["mean"].min(), 1e-9), 2))

In [ ]:
# EDA 6: relation with covariates: binned means and Spearman correlation (robust to outliers)
from scipy.stats import spearmanr
rows = []
for c in [c for c in EXOG_COLS if c in df.columns]:
    ok = df[[c, TARGET_COL]].dropna(); rho, p = spearmanr(ok[c], ok[TARGET_COL]); rows.append([c, round(rho, 3), p, len(ok)])
    fig, ax = plt.subplots(figsize=(5.5, 3.2)); b = ok.groupby(pd.qcut(ok[c], 10, duplicates="drop"))[TARGET_COL].mean()
    ax.plot(range(len(b)), b.values, marker="o", color=TEAL); ax.set_xlabel(c + " (deciles)"); ax.set_ylabel("mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_covariate_" + c)
corr_tbl = pd.DataFrame(rows, columns=["covariate", "spearman_rho", "p_value", "n"]); corr_tbl.to_csv("report/table_covariate_corr.csv", index=False); display(corr_tbl)

In [ ]:
# EDA 7: cross-correlation with lags: does the covariate lead the target? (positive lag = covariate earlier)
def cross_corr(g, x, y, lags):
    return [g[y].corr(g[x].shift(k)) for k in lags]
lags = list(range(0, 4 * SEASON + 1, max(1, SEASON // 4)))
fig, ax = plt.subplots(figsize=(7, 3.2))
for c in [c for c in EXOG_COLS if c in df.columns][:3]:
    cc = np.nanmean([cross_corr(g, c, TARGET_COL, lags) for _, g in df.groupby(UNIT_COL) if len(g) > 5 * SEASON], axis=0)
    ax.plot(lags, cc, marker="o", label=c); print(c, "best lag:", lags[int(np.nanargmax(np.abs(cc)))], "corr:", round(float(np.nanmax(np.abs(cc))), 3))
ax.axhline(0, color=GREY); ax.set_xlabel("lag (steps, covariate leads target)"); ax.set_ylabel("correlation"); ax.legend(frameon=False); style(ax); savefig(fig, "fig_eda_crosscorr")

In [ ]:
# EDA 8: autocorrelation of the target (ACF) averaged over units: which lags to use as features and whether SEASON is right
from statsmodels.tsa.stattools import acf
nl = 2 * SEASON + 1
acfs = [acf(g[TARGET_COL].values, nlags=nl, fft=True) for _, g in df.groupby(UNIT_COL) if len(g) > 4 * SEASON]
acf_mean = np.mean(acfs, axis=0)
fig, ax = plt.subplots(figsize=(8, 3.2)); ax.stem(range(nl + 1), acf_mean, basefmt=" "); ax.set_xlabel("lag (steps)"); ax.set_ylabel("ACF"); style(ax); savefig(fig, "fig_eda_acf")
print("ACF at lag 1:", round(acf_mean[1], 3), "| at SEASON:", round(acf_mean[SEASON], 3), "-> a seasonal naive baseline is strong if ACF at SEASON > 0.7")

In [ ]:
# EDA 9: missingness pattern over time per unit (heatmap of monthly missing share) and regime check (rolling mean over time)
miss = df.set_index(TIME_COL).groupby(UNIT_COL)[TARGET_COL].resample("MS").apply(lambda s: s.isna().mean()).unstack(0)
fig, ax = plt.subplots(figsize=(10, 3.4)); sns.heatmap(miss.T, cmap="Blues", cbar_kws={"label": "missing share"}, ax=ax); ax.set_xlabel("month"); ax.set_ylabel(UNIT_COL); savefig(fig, "fig_eda_missing")
roll = df.set_index(TIME_COL)[TARGET_COL].resample("MS").mean()
fig, ax = plt.subplots(figsize=(10, 3.0)); ax.plot(roll.index, roll.values, color=TEAL, marker="o", ms=3); ax.axvline(pd.Timestamp(TEST_START), color="k", ls="--"); ax.set_ylabel("monthly mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_regime")
print("largest month-to-month change (share of mean):", round((roll.diff().abs().max() / roll.mean()), 3), "-> above 0.5 suggests a regime shift worth reporting in RQ3")

### Ghi nhận xét EDA (bắt buộc, dùng cho phần Results RQ1)

Điền 5 nhận xét, mỗi nhận xét chỉ đúng một con số ở bảng hoặc hình phía trên, ví dụ: "Sức mạnh mùa theo giờ là 0,42 trong khi theo tháng chỉ 0,08, nên đặc trưng giờ quan trọng hơn tháng (Hình EDA 4)". Không viết nhận xét không có số.

In [ ]:
# EDA 10: write your five evidence-backed observations here (they go straight into the paper)
eda_notes = [
    "1. ...",
    "2. ...",
    "3. ...",
    "4. ...",
    "5. ...",
]
open("report/eda_notes.md", "w").write("\n".join(eda_notes)); print("saved report/eda_notes.md")

## Bước 3: SQL với DuckDB, trả lời RQ1 và tạo bảng đặc trưng

**Vì sao SQL**: người đọc bài kiểm tra được từng con số bằng một truy vấn; đặc trưng trễ và cửa sổ trượt bằng `WINDOW` không rò rỉ tương lai nếu chỉ dùng `LAG` và `PRECEDING`.

**Quy tắc chống rò rỉ**: đặc trưng chỉ dùng thông tin **tại hoặc trước** thời điểm t; mục tiêu là `LEAD(target, h)`; không dùng thống kê tính trên toàn chuỗi (kể cả trung bình đơn vị) trừ khi tính chỉ trên phần huấn luyện.

In [ ]:
# Step 3a: register the clean table and write the RQ1 queries to sql/queries.sql (edit the four queries to your topic; keep them minimal and readable)
con.register("clean", df)
EXOG_SQL = ", ".join([c for c in EXOG_COLS if c in df.columns]) or "NULL AS no_exog"
SQL_RQ1 = f"""
-- Q1: mean and spread of the target per unit and per calendar period
SELECT {UNIT_COL}, month({TIME_COL}) AS mon, AVG({TARGET_COL}) AS mean_target, STDDEV({TARGET_COL}) AS sd_target, COUNT(*) AS n FROM clean GROUP BY 1, 2 ORDER BY 1, 2;
-- Q2: share of event steps (target above the {EVENT_QUANTILE:.0%} quantile) per unit
WITH thr AS (SELECT quantile_cont({TARGET_COL}, {EVENT_QUANTILE}) AS q FROM clean)
SELECT {UNIT_COL}, AVG(({TARGET_COL} > q)::INT) AS event_share FROM clean, thr GROUP BY 1 ORDER BY 2 DESC;
-- Q3: target by covariate decile (relationship with the secondary source)
SELECT decile, AVG({TARGET_COL}) AS mean_target, COUNT(*) AS n FROM (SELECT NTILE(10) OVER (ORDER BY {EXOG_COLS[0] if EXOG_COLS else TARGET_COL}) AS decile, {TARGET_COL} FROM clean WHERE {EXOG_COLS[0] if EXOG_COLS else TARGET_COL} IS NOT NULL) GROUP BY 1 ORDER BY 1;
-- Q4: seasonal-naive skill check: correlation between the target and its value SEASON steps earlier
SELECT corr({TARGET_COL}, lagged) AS r_season FROM (SELECT {TARGET_COL}, LAG({TARGET_COL}, {SEASON}) OVER (PARTITION BY {UNIT_COL} ORDER BY {TIME_COL}) AS lagged FROM clean);
"""
open("sql/queries.sql", "w").write(SQL_RQ1); print(SQL_RQ1)

In [ ]:
# Step 3b: run every statement, save each result as report/table_q{k}.csv (comment lines removed before splitting on ';')
def run_sql_file(path):
    txt = open(path).read(); body = "\n".join(l for l in txt.splitlines() if not l.strip().startswith("--"))
    k = 0
    for stmt in body.split(";"):
        stmt = stmt.strip()
        if not stmt: continue
        k += 1; out = con.execute(stmt).df(); out.to_csv(f"report/table_q{k}.csv", index=False); print(f"Q{k}: {out.shape}"); display(out.head(8))
run_sql_file("sql/queries.sql")

In [ ]:
# Step 3c: statistical tests behind RQ1 (report p-values, not only means): Kruskal-Wallis across units, Spearman with covariates
from scipy.stats import kruskal
groups = [g[TARGET_COL].values for _, g in df.groupby(UNIT_COL) if len(g) > 30]
H, p = kruskal(*groups) if len(groups) > 1 else (np.nan, np.nan)
tests = [["Kruskal-Wallis target across units", round(H, 2) if H == H else "n/a", p]]
for c in [c for c in EXOG_COLS if c in df.columns]:
    ok = df[[c, TARGET_COL]].dropna(); rho, pv = spearmanr(ok[c], ok[TARGET_COL]); tests.append([f"Spearman target vs {c}", round(rho, 3), pv])
tests = pd.DataFrame(tests, columns=["test", "statistic", "p_value"]); tests.to_csv("report/table_rq1_tests.csv", index=False); display(tests)

In [ ]:
# Step 3d: build the feature table with SQL (lags, rolling means, calendar, covariates, targets for every horizon); no future information
lag_list = sorted(set([1, 2, 3, SEASON, 2 * SEASON, 7 * SEASON if FREQ in ("h", "H") else SEASON * 4]))
lag_sql = ", ".join([f"LAG({TARGET_COL}, {k}) OVER w AS y_lag{k}" for k in lag_list])
roll_sql = f"AVG({TARGET_COL}) OVER (w ROWS BETWEEN {SEASON - 1} PRECEDING AND CURRENT ROW) AS y_ma_season, STDDEV({TARGET_COL}) OVER (w ROWS BETWEEN {SEASON - 1} PRECEDING AND CURRENT ROW) AS y_sd_season, " \
           f"{TARGET_COL} - LAG({TARGET_COL}, 1) OVER w AS y_diff1"
exog_lag = ", ".join([f"LAG({c}, {SEASON}) OVER w AS {c}_lag_season" for c in EXOG_COLS if c in df.columns])
targets = ", ".join([f"LEAD({TARGET_COL}, {h}) OVER w AS y_h{h}" for h in HORIZONS])
cal_sql = f"hour({TIME_COL}) AS hr, dayofweek({TIME_COL}) AS dow, month({TIME_COL}) AS mon, dayofyear({TIME_COL}) AS doy"
FEAT_SQL = f"""
CREATE OR REPLACE TABLE feat AS
SELECT {UNIT_COL}, {TIME_COL}, {TARGET_COL}, {EXOG_SQL}, {cal_sql}, {lag_sql}, {roll_sql}{', ' + exog_lag if exog_lag else ''}, {targets}
FROM clean WINDOW w AS (PARTITION BY {UNIT_COL} ORDER BY {TIME_COL});
"""
con.execute(FEAT_SQL); open("sql/features.sql", "w").write(FEAT_SQL)
feat = con.execute("SELECT * FROM feat").df(); print(feat.shape); display(feat.head(3))

In [ ]:
# Step 3e: leakage check: for every row, the largest lag feature must come from a timestamp strictly before the target timestamp (by construction) and
# the target of horizon h must equal the raw target h steps later within the same unit
g0 = feat[feat[UNIT_COL] == feat[UNIT_COL].iloc[0]].sort_values(TIME_COL).reset_index(drop=True)
h = HORIZONS[0]; ok = np.allclose(g0[f"y_h{h}"].iloc[:-h].values, g0[TARGET_COL].iloc[h:].values, equal_nan=True)
print("horizon target aligned with raw target shifted by h:", ok)
assert ok, "target alignment broken: check FREQ and the resample grid"

In [ ]:
# Step 3f: RQ1 figures 2 and 3 for the paper: event share per unit and target by covariate decile (from the SQL outputs)
q2 = pd.read_csv("report/table_q2.csv"); q3 = pd.read_csv("report/table_q3.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
b = axes[0].bar(range(len(q2)), q2.event_share, color=TEAL); axes[0].set_xlabel("units"); axes[0].set_ylabel("event share"); style(axes[0])
axes[1].plot(q3.decile, q3.mean_target, marker="o", color=ACC); axes[1].set_xlabel("covariate decile"); axes[1].set_ylabel("mean " + TARGET_COL); style(axes[1])
savefig(fig, "fig_rq1_sql")

## Kiểm tra, bàn giao và ghi Audit Log

**Test case** phải chạy qua trước khi bàn giao. **Manifest** ghi số dòng, cột, khoảng thời gian và mã MD5 của parquet để Sinh viên B xác nhận nhận đúng file. Sau đó điền **AI Audit Log** cho các prompt đã dùng ở tuần 1 và 2 (ít nhất 3 mục mỗi tuần, có ít nhất một mục phát hiện AI sai).

In [ ]:
# Test cases for Notebook_C (all asserts must pass)
assert feat[[UNIT_COL, TIME_COL]].duplicated().sum() == 0, "duplicate unit-time rows"
assert feat[TARGET_COL].notna().all(), "target missing after cleaning"
assert set(HORIZONS) <= set(int(c[3:]) for c in feat.columns if c.startswith("y_h")), "horizon targets missing"
assert (feat.groupby(UNIT_COL)[TIME_COL].apply(lambda s: s.is_monotonic_increasing)).all(), "time not sorted within unit"
assert len(feat) >= 30000, f"only {len(feat)} rows: collect more data (Step 1d) or lower FREQ"
print("Tests C: OK")

In [ ]:
# Handover: parquet + manifest with md5
feat.to_parquet("data/processed/feat.parquet", index=False)
feat.to_csv("data/processed/feat.csv", index=False)                    # CSV copy of the handover table (parquet is the file of record)
md5 = hashlib.md5(open("data/processed/feat.parquet", "rb").read()).hexdigest()
manifest = {"group": GROUP, "topic": TOPIC, "rows": len(feat), "cols": list(feat.columns), "units": int(feat[UNIT_COL].nunique()), "start": str(feat[TIME_COL].min()), "end": str(feat[TIME_COL].max()),
            "freq": FREQ, "horizons": HORIZONS, "season": SEASON, "unit_col": UNIT_COL, "time_col": TIME_COL, "target_col": TARGET_COL, "exog_cols": [c for c in EXOG_COLS if c in feat.columns], "test_start": TEST_START, "md5": md5}
json.dump(manifest, open("data/processed/manifest.json", "w"), indent=2); print(json.dumps(manifest, indent=2)[:600])

In [ ]:
# Record the prompts you used this week (edit the examples; keep only real prompts)
audit("Step 1d", f"Đề tài {TOPIC}: gợi ý nguồn phụ ghép được theo {TIME_COL}", "Claude", "3 nguồn được gợi ý", "mở URL, kiểm tra giấy phép", "chọn nguồn ..., loại nguồn ... vì ...")
audit("Step 2c", f"Cột {TARGET_COL} thiếu {round(float(gap.mean()), 3)} theo cụm; nội suy hay bỏ?", "Claude", "khuyên interpolate(limit=3)", "kiểm tra không dùng giá trị tương lai", "áp dụng limit=3")
print(pd.read_csv(AUDIT_PATH).tail(3))

## Bài tập mở rộng cho Sinh viên A (làm ít nhất 3 trong 5, ghi kết quả vào report/exercises_C.md)

1. **Nguồn thứ ba**: thêm một nguồn phụ nữa (lịch ngày lễ, giá, sự kiện) bằng cùng hàm `load_table` và `merge_asof`; báo tỷ lệ dòng ghép được và tương quan với mục tiêu.
2. **So sánh tần suất**: gộp mục tiêu về tần suất thô hơn (ví dụ ngày) và so sức mạnh mùa giữa hai tần suất; kết luận tần suất nào phù hợp cho câu hỏi của đề.
3. **Hai truy vấn SQL thêm**: một truy vấn dùng `QUALIFY` để lấy top 5 đơn vị theo mỗi tháng, một truy vấn `WITH` tính tỷ lệ sự kiện theo mùa; giải thích kết quả bằng hai câu.
4. **Phân rã mùa**: dùng `statsmodels.tsa.seasonal.STL` trên một đơn vị; vẽ trend, seasonal, resid; nhận xét phần dư có còn cấu trúc không.
5. **Kiểm tra rò rỉ có chủ ý**: tạo một đặc trưng sai (dùng `LEAD`) rồi cho Sinh viên B chạy thử để thấy MAE giảm bất thường; ghi lại bài học vào Discussion.

In [ ]:
# Exercise 4 starter: STL decomposition for one unit (period = SEASON)
from statsmodels.tsa.seasonal import STL
u0 = df[UNIT_COL].unique()[0]; s = df[df[UNIT_COL] == u0].set_index(TIME_COL)[TARGET_COL].asfreq(FREQ).interpolate(limit=3).dropna()
if len(s) > 3 * SEASON:
    stl = STL(s, period=SEASON, robust=True).fit()
    fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
    for ax, comp, lab in zip(axes, [stl.trend, stl.seasonal, stl.resid], ["trend", "seasonal", "residual"]): ax.plot(comp, color=TEAL, lw=0.8); ax.set_ylabel(lab); style(ax)
    savefig(fig, "fig_exercise_stl"); print("residual share of variance:", round(float(stl.resid.var() / s.var()), 3))

In [ ]:
# Exercise 2 starter: seasonal strength at a coarser frequency (daily) versus the working frequency
daily = df.set_index(TIME_COL).groupby(UNIT_COL)[TARGET_COL].resample("D").mean().reset_index()
strength = lambda frame, key: round(frame.groupby(key)[TARGET_COL].mean().std() / frame[TARGET_COL].std(), 3)
print("weekday strength: working freq", strength(df.assign(k=df[TIME_COL].dt.dayofweek), "k"), "| daily", strength(daily.assign(k=daily[TIME_COL].dt.dayofweek), "k"))

In [ ]:
# Exercise 3 starter: QUALIFY and CTE examples (edit and add your own two queries to sql/queries.sql)
ex_sql = f"""
SELECT {UNIT_COL}, month({TIME_COL}) AS mon, AVG({TARGET_COL}) AS m, RANK() OVER (PARTITION BY month({TIME_COL}) ORDER BY AVG({TARGET_COL}) DESC) AS rk FROM clean GROUP BY 1, 2 QUALIFY rk <= 5 ORDER BY mon, rk;
"""
display(con.execute(ex_sql).df().head(10))

### Tự đánh giá Notebook_C (điền trước khi gửi giảng viên)

| Tiêu chí | Tự chấm (0-2) | Bằng chứng |
|---|---|---|
| Nguồn và giấy phép rõ ràng | | report/data_sources.csv |
| So sánh hai nguồn có số | | table_source_comparison.csv |
| Làm sạch có nhật ký | | table_cleaning_log.csv |
| EDA có nhận xét kèm số | | eda_notes.md |
| SQL không rò rỉ, có kiểm tra | | features.sql, ô kiểm tra rò rỉ |
| Bài tập mở rộng | | exercises_C.md |
| AI Audit Log | | ai_audit_log.csv |

## Lỗi thường gặp và cách sửa

| Lỗi | Nguyên nhân | Cách sửa |
|---|---|---|
| `KeyError: 'ts'` | tên cột trong file khác CONFIG | in `df.columns`, sửa `TIME_COL` |
| Rò rỉ tương lai (điểm quá tốt) | dùng `shift(-k)` hoặc trung bình cả chuỗi khi tạo đặc trưng | chỉ dùng `LAG` và cửa sổ `PRECEDING`; kiểm tra `feat.ts` của đặc trưng luôn nhỏ hơn mục tiêu |
| Trùng dòng theo đơn vị và thời gian | nguồn có hai bản ghi cùng thời điểm | `drop_duplicates([UNIT_COL, TIME_COL])` rồi ghi vào nhật ký làm sạch |
| Lưới thời gian thiếu bước | không `resample(FREQ)` | resample và đếm bước thiếu theo đơn vị |
| Ghép nguồn phụ mất nửa dòng | lệch múi giờ hoặc lệch tần suất | đưa cả hai về UTC hoặc giờ địa phương thống nhất, dùng `merge_asof` với `tolerance` |
| DuckDB `Binder Error` | tên cột là từ khoá (`do`, `at`, `year`) hoặc trùng tên bảng | đổi tên cột, dùng dấu ngoặc kép |
| `MemoryError` | nạp toàn bộ file lớn bằng pandas | dùng `duckdb.read_csv_auto` và lọc sớm |
| Chronos không cài được | thiếu `pip install chronos-forecasting` | bỏ qua ô mô hình nền, ghi vào hạn chế |

## Checklist Notebook_C trước khi bàn giao (tick từng dòng)

- [ ] Ba file CSV: `data/raw/primary_snapshot.csv`, `data/processed/clean.csv`, `data/processed/feat.csv` mở được bằng Excel, số dòng khớp manifest
- [ ] `report/data_sources.csv` có đủ hai nguồn với URL, giấy phép, số dòng, khoảng thời gian
- [ ] `report/table_source_comparison.csv` và một câu nhận xét về mức nhất quán giữa hai nguồn
- [ ] `report/table_cleaning_log.csv` giải thích mọi dòng bị bỏ
- [ ] 9 hình EDA có caption nêu số liệu; `report/eda_notes.md` có 5 nhận xét kèm số
- [ ] `sql/queries.sql` 4 truy vấn RQ1 chạy qua; `report/table_q1..q4.csv`; `report/table_rq1_tests.csv` có p-value
- [ ] `sql/features.sql` không dùng LEAD ngoài mục tiêu; ô kiểm tra rò rỉ qua
- [ ] Test cases qua; `feat.parquet` và `manifest.json` gửi cho Sinh viên B
- [ ] AI Audit Log có ít nhất 6 mục cho tuần 1 và 2, ít nhất một mục `hallucination=True`

## Mẫu viết phần Data và RQ1 cho bài báo (điền số từ các bảng)

**Data.** We use {PRIMARY_SOURCE} ({license}, {rows} rows, {units} units, {start} to {end}, frequency {FREQ}) merged with {SECOND_SOURCE} ({share}% of rows have covariates). Cleaning removed {removed} rows (Table cleaning log); the two sources agree on {metric} (Table source comparison).

**RQ1.** The target varies most with {calendar or covariate}: seasonal strength {value} by {period} versus {value} by {period} (Figure EDA 4); units differ by a factor of {ratio} (Figure EDA 5); the covariate {name} leads the target by {lag} steps with correlation {rho} (Figure EDA 7); autocorrelation at the seasonal lag is {acf}, so the seasonal naive baseline is expected to be strong (Figure EDA 8). Kruskal-Wallis across units gives p = {p} (Table RQ1 tests).